<a href="https://colab.research.google.com/github/manuelagutierrezss16/Integraci-n-de-Datos-y-Prospectiva/blob/main/4_INTEGRACI%C3%93N_MULTIDIMENSIONAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CASO DE ESTUDIO**

Una entidad prestadora de servicios de salud (EPS) del sistema de
atención en salud en Colombia, requiere mejorar la eficiencia de sus
operaciones del negocio, por lo cual debe reducir su tamaño. (optimización de operaciones integrando diferentes sucursales ubicadas en el Valle de Aburra.)
En este sentido, la EPS quiere centrar sus operaciones en los sectores más cercanos a la ciudad de Medellín por lo que quiere hacer la integración de los pacientes de Coopacabana y Caldas en sus otras sedes. Dentro de los objetivos, quiere evaluar como era la configuración de sus variables para antes y despues de la integracíon de pacientes nuevos.

0. Carga librerias de trabajo

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


1. Cargamos la base de datos de referencia y creamos las bases de datos de integración

In [ ]:
nxl='/content/drive/MyDrive/Colab Notebooks/INTEGRACION/BASES DE DDATOS/5. Diabetes Árbol_Int_Mult.xlsx'
XDB=pd.read_excel (nxl,sheet_name=0)
XDB=XDB.dropna()
XDB.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome,Numero_Atenciones,Costo_Promedio_Atencion,Ciudad_Pertenencia
0,0,125,96,0,0,22.5,0.262,21,0,4,314.926044,Sabaneta
1,0,141,0,0,0,42.4,0.205,29,1,6,287.355812,Envigado
2,10,101,86,37,0,45.6,1.136,38,1,8,464.541158,Sabaneta
3,1,96,122,0,0,22.4,0.207,27,0,3,368.103292,Sabaneta
4,5,139,64,35,140,28.6,0.411,26,0,9,428.548590,Copacabana


In [ ]:
# SE CONSTRUYE LA PRIMERA BASE DE DATOS, SOLO CON VARIABLES DE ENTRADA (XD) (se elimina outcome) PARA CREAR CLUSTERS SIN CIUDAD Y OUTCOME
XD = XDB.iloc[:,[0,1,2,3,4,5,6,7,9,10]]
XD.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Numero_Atenciones,Costo_Promedio_Atencion
0,0,125,96,0,0,22.5,0.262,21,4,314.926044
1,0,141,0,0,0,42.4,0.205,29,6,287.355812
2,10,101,86,37,0,45.6,1.136,38,8,464.541158
3,1,96,122,0,0,22.4,0.207,27,3,368.103292
4,5,139,64,35,140,28.6,0.411,26,9,428.548590


In [ ]:
#SE CONSTRUYE LA SEGUNDA BASE DE DATOS (XDB2) PARA CLASIFICAR POR DIABETES Y CIUDAD
XDB2=XDB.iloc[:,[0,1,2,3,4,5,6,7,8,9,10,11]]
XDB2.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome,Numero_Atenciones,Costo_Promedio_Atencion,Ciudad_Pertenencia
0,0,125,96,0,0,22.5,0.262,21,0,4,314.926044,Sabaneta
1,0,141,0,0,0,42.4,0.205,29,1,6,287.355812,Envigado
2,10,101,86,37,0,45.6,1.136,38,1,8,464.541158,Sabaneta
3,1,96,122,0,0,22.4,0.207,27,0,3,368.103292,Sabaneta
4,5,139,64,35,140,28.6,0.411,26,0,9,428.548590,Copacabana


SE PUEDE HACER CON LA BASE DE DATOS PRINCIPAL LLAMANDO A CADA COSA QUE NECESITO, SE ORGANIZA ASÍ POR ESTETICA Y LOGICA

2. Se procede con la creación de clusters

In [ ]:
# Numero de cluster a los que va a estar asociado cada dato
nc=np.zeros((len(XD),1))
#XC van a ser los 5 primeros datos de la base de datos inicial, no son clusters sino semillas del metodo kmedoits
XD=np.array (XD);
XC=XD[0:5,:] # Correctly slice the NumPy array XD
XC=np.array(XC)

#Como se clusteriza un dato?
#XC-XD[0,:] # distancia primer dato con cada una de las 5 semillas (la primera semilla es el primer dato)
#XC-XD[1,:] # distancia segundo dato con cada una de las 5 semillas(los ceros estan en el segundo ya que es el mismo)

for k in range (len(XD)):
  d=np.sqrt(np.sum((XC-XD[k,:])**2,axis=1))
  nc[k,]=int(np.argmin(d))
  cl=int(np.argmin(d)) # se usa temporal mente, se guarda nc
  #print(nc[k,])

  XC[cl,:]=(XC[cl,:]+XD[k,:])/2
XD=XDB.iloc[:,[0,1,2,3,4,5,6,7,9,10]]
XC2=pd.DataFrame(XC,columns=XD.columns)
display(XC2)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Numero_Atenciones,Costo_Promedio_Atencion
0,3.243770,117.992289,72.022205,24.348928,128.132845,32.159844,0.380072,29.691719,5.845113,245.809207
1,3.171596,123.559375,71.998238,40.203914,50.893302,30.232954,0.354780,32.960593,6.215968,122.832710
2,1.789979,160.538699,75.414111,32.855127,579.512355,44.537231,0.447227,26.771967,3.821158,240.893389
3,4.164665,153.096176,69.457456,38.513445,243.003131,42.662530,0.540859,30.184062,5.238367,328.992879
4,7.915420,115.692465,64.699479,19.252366,0.298011,35.946307,0.682575,42.917586,6.285666,404.000258


In [ ]:
from pandas.io.sql import PandasSQL
XDB2['Cluster']=nc #Adiciono a esta tabla la columna cluster
XDB2.head()
NPD= XDB2. groupby('Cluster').agg({'Outcome':lambda x: (x==1).sum()}) #digame por cada cluster cuantos pacientes tienen diabetes, la variable outcome se volvio x
NPND= XDB2. groupby('Cluster').agg({'Outcome':lambda x: (x==0).sum()}) #digame por cada cluster cuantos pacientes NO tienen diabetes, la variable outcome se volvio x

pdi=NPD/(NPD+NPND)
pnd=NPND/(NPD+NPND)

#EL DATAFRAME CON LOS CLUSTERS ES EL SIGUIENTE

XC2['Diabetes']=pdi;XC2['No Diabetes']=pnd
display(XC2)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Numero_Atenciones,Costo_Promedio_Atencion,Diabetes,No Diabetes
0,3.243770,117.992289,72.022205,24.348928,128.132845,32.159844,0.380072,29.691719,5.845113,245.809207,0.314714,0.685286
1,3.171596,123.559375,71.998238,40.203914,50.893302,30.232954,0.354780,32.960593,6.215968,122.832710,0.398721,0.601279
2,1.789979,160.538699,75.414111,32.855127,579.512355,44.537231,0.447227,26.771967,3.821158,240.893389,0.324519,0.675481
3,4.164665,153.096176,69.457456,38.513445,243.003131,42.662530,0.540859,30.184062,5.238367,328.992879,0.370262,0.629738
4,7.915420,115.692465,64.699479,19.252366,0.298011,35.946307,0.682575,42.917586,6.285666,404.000258,0.379182,0.620818


ANALISIS

En el cluster 1 (el segundo) se puede observar el mayor porcentaje de personas con diabetes, esto promovido principlamente por la insulina baja (50,89), se destaca en este mismo cluster un numero elevado de atenciones por año para cafa paciente (6,2) lo que refuerza la precensia de diabetes en este grupo de pacientes.Se destaca el cluster 4, el cual posee el valor de insulina más bajo,y el segundo valor más alto de pacientes con diabetes, con edades mucho más altas y con un numero de embarazos mayor